<a href="https://colab.research.google.com/github/Habibaaboalhassan66/swimming-detection/blob/main/posestimationbutterfly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests numpy tqdm -q
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
"""
File 2: pose_estimation.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Extracts 7 upper-body keypoints per frame using fine-tuned Roboflow models.
Reads stroke detection results from Drive. Saves pose results to Drive.

CHANGE FROM PREVIOUS VERSION:
- Reads/writes from Google Drive paths, not /content/
- Runs Roboflow on PREPROCESSED frames from per-video subfolders:
    breaststroke_testing/{video_name}/frame_000000.jpg ...
    butterfly_testing (1)/{video_name}/frame_000000.jpg ...
"""

import os
import json
import base64
import requests
import numpy as np
import cv2
import math
from pathlib import Path
from google.colab import drive

# ─── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ─── Paths (all Drive-based so they survive session restarts) ──────────────────
STROKE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/stroke_detection_results.json"
POSE_RESULTS_PATH   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results.json"
VIZ_DIR             = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations"

BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"
BUTTERFLY_TEST_DIR    = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"

# ─── Roboflow config ───────────────────────────────────────────────────────────
API_KEY        = "2gU1ewQS0rfbADpO4tCb"
WORKSPACE      = "habibas-workspace-jcdgt"
BREASTSTROKE_MODEL = "swimmer-breaststroke-front1/2"
BUTTERFLY_MODEL    = "swimmer-butterfly-front1/4"
ROBOFLOW_INFER_URL = "https://detect.roboflow.com/{model}"

# ─── Settings ─────────────────────────────────────────────────────────────────
SAMPLE_SIZE         = 300
KEYPOINT_THRESHOLD  = 0.70
KEYPOINT_NAMES      = ["head", "left_shoulder", "right_shoulder",
                       "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

# Visualization colors
COLOR_CONFIDENT  = (0, 255, 0)    # green
COLOR_OCCLUDED   = (128, 128, 128) # grey
COLOR_SKELETON   = (0, 0, 0)       # black
HALO_COLOR       = (0, 200, 0)

# Skeleton connections (index pairs into KEYPOINT_NAMES)
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]


def load_stroke_results():
    """Load File 1 output from Drive."""
    if not os.path.exists(STROKE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Stroke results not found at {STROKE_RESULTS_PATH}. "
            "Run File 1 (stroke_detection_cnn.py) first."
        )
    with open(STROKE_RESULTS_PATH, 'r') as f:
        return json.load(f)


def get_video_dirs_by_stroke(stroke_results: dict) -> dict:
    """
    Map each video name to its preprocessed frame subfolder and stroke type.
    Reads 'detected_stroke' from File 1 output.
    Builds the folder path using our own directory constants (not folder_path
    from File 1, which may have stale/renamed paths).
    Returns: {video_name: (stroke_type, video_subfolder_path)}
    """
    mapping = {}
    for video_name, result in stroke_results.items():
        stroke = result.get("detected_stroke", "unknown")
        if stroke == "breaststroke":
            folder_path = os.path.join(BREASTSTROKE_TEST_DIR, video_name)
            mapping[video_name] = (stroke, folder_path)
        elif stroke == "butterfly":
            folder_path = os.path.join(BUTTERFLY_TEST_DIR, video_name)
            mapping[video_name] = (stroke, folder_path)
        else:
            print(f"  [SKIP] {video_name} — unknown stroke '{stroke}'")
    return mapping


def select_frames(all_frames: list, stroke: str) -> list:
    """
    Select a representative subset of frames based on stroke type.
    Breaststroke: middle 80% (skip first/last 10%)
    Butterfly:    last 45% (skip first 55% — entry phase)
    Then sample up to SAMPLE_SIZE evenly.
    """
    n = len(all_frames)
    if stroke == "breaststroke":
        start = int(n * 0.10)
        end   = int(n * 0.90)
    else:  # butterfly
        start = int(n * 0.10)
        end   = int(n * 0.90)

    subset = all_frames[start:end]
    if len(subset) <= SAMPLE_SIZE:
        return subset

    # Evenly spaced sample
    indices = np.linspace(0, len(subset) - 1, SAMPLE_SIZE, dtype=int)
    return [subset[i] for i in indices]


def encode_image_b64(image_path: str) -> str:
    """Read image file and return base64 string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_roboflow_keypoint(image_b64: str, model_id: str) -> dict | None:
    """
    Call Roboflow REST API (NOT inference-sdk — broken in Colab).
    Returns raw response JSON or None on failure.
    """
    url = f"https://detect.roboflow.com/{model_id}?api_key={API_KEY}"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    try:
        resp = requests.post(url, data=image_b64, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"    [API ERROR] {e}")
        return None


def parse_keypoints(api_response: dict, img_w: int, img_h: int) -> dict:
    """
    Extract 7 keypoints from Roboflow response.
    Returns dict: {keypoint_name: {x, y, confidence, visible}}
    Falls back to estimated positions for missing keypoints.
    """
    keypoints = {}

    predictions = api_response.get("predictions", [])
    if not predictions:
        return {kp: {"x": None, "y": None, "confidence": 0.0, "visible": False}
                for kp in KEYPOINT_NAMES}

    # Take the highest-confidence prediction (the swimmer)
    best = max(predictions, key=lambda p: p.get("confidence", 0))
    raw_kps = {kp.get("class", "").lower(): kp
               for kp in best.get("keypoints", [])}

    for kp_name in KEYPOINT_NAMES:
        if kp_name in raw_kps:
            kp = raw_kps[kp_name]
            conf = float(kp.get("confidence", 0.0))
            x    = float(kp.get("x", 0))
            y    = float(kp.get("y", 0))
            keypoints[kp_name] = {
                "x": x, "y": y,
                "confidence": conf,
                "visible": conf >= KEYPOINT_THRESHOLD
            }
        else:
            keypoints[kp_name] = {
                "x": None, "y": None,
                "confidence": 0.0,
                "visible": False
            }

    # ── Geometric validation ──────────────────────────────────────────────────
    head = keypoints.get("head")
    ls   = keypoints.get("left_shoulder")
    rs   = keypoints.get("right_shoulder")

    # Head must be above shoulders (lower y = higher in image)
    if head["visible"] and ls["visible"] and rs["visible"]:
        shoulder_y = (ls["y"] + rs["y"]) / 2
        if head["y"] > shoulder_y:
            head["visible"] = False  # fail geometric check

    # Elbows within 1.5× shoulder width
    if ls["visible"] and rs["visible"]:
        sw = abs(ls["x"] - rs["x"])
        for side, elbow_name in [("left", "left_elbow"), ("right", "right_elbow")]:
            elbow = keypoints.get(elbow_name)
            if elbow["visible"] and sw > 0:
                ref_x = ls["x"] if side == "left" else rs["x"]
                if abs(elbow["x"] - ref_x) > 1.5 * sw:
                    elbow["visible"] = False

    # Wrists within 1.5× shoulder width of their elbow
    for side, wrist_name, elbow_name in [
        ("left", "left_wrist", "left_elbow"),
        ("right", "right_wrist", "right_elbow")
    ]:
        wrist = keypoints.get(wrist_name)
        elbow = keypoints.get(elbow_name)
        if wrist["visible"] and elbow["visible"] and ls["visible"] and rs["visible"]:
            sw = abs(ls["x"] - rs["x"])
            if abs(wrist["x"] - elbow["x"]) > 1.5 * sw:
                wrist["visible"] = False

    # ── Wrist mirroring: if both wrists overlap, mirror the wrong one ─────────
    lw = keypoints.get("left_wrist")
    rw = keypoints.get("right_wrist")
    if lw["visible"] and rw["visible"] and lw["x"] is not None and rw["x"] is not None:
        dist = math.sqrt((lw["x"] - rw["x"])**2 + (lw["y"] - rw["y"])**2)
        if dist < 30:
            # Mirror: reflect wrong wrist across shoulder midpoint
            if ls["visible"] and rs["visible"]:
                mid_x = (ls["x"] + rs["x"]) / 2
                # Move right wrist to mirror of left wrist
                rw["x"] = 2 * mid_x - lw["x"]

    return keypoints


def compute_measurements(keypoints: dict) -> dict:
    """
    Compute the 3 biomechanical measurements used for injury pattern detection.
    Returns dict with measurements and validity flags.
    """
    ls = keypoints.get("left_shoulder")
    rs = keypoints.get("right_shoulder")
    hd = keypoints.get("head")

    measurements = {
        "shoulder_asymmetry": None,     # abs(left_shoulder_y - right_shoulder_y)
        "shoulder_width": None,         # abs(left_shoulder_x - right_shoulder_x)
        "head_offset": None,            # abs(head_x - shoulder_midpoint_x)
        "valid": False
    }

    if ls["visible"] and rs["visible"] and ls["x"] is not None:
        measurements["shoulder_asymmetry"] = abs(ls["y"] - rs["y"])
        measurements["shoulder_width"]     = abs(ls["x"] - rs["x"])
        measurements["valid"] = True

        if hd["visible"] and hd["x"] is not None:
            mid_x = (ls["x"] + rs["x"]) / 2
            measurements["head_offset"] = abs(hd["x"] - mid_x)

    return measurements


def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """Draw keypoints and skeleton on image. Returns annotated copy."""
    img = image.copy()
    kp_coords = {}

    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y)
        visible = kp.get("visible", False)

        if visible:
            # Halo effect
            cv2.circle(img, (x, y), 12, HALO_COLOR, -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 8, COLOR_CONFIDENT, -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 6, COLOR_OCCLUDED, -1, cv2.LINE_AA)

    # Draw skeleton connections
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            cv2.line(img, kp_coords[n1], kp_coords[n2], COLOR_SKELETON, 2, cv2.LINE_AA)

    return img


def process_video_frames(video_name: str, stroke: str, frames_dir: str) -> dict:
    """
    Process a single video's frames:
    1. List and select frames
    2. Call Roboflow for each frame
    3. Parse keypoints + compute measurements
    4. Optionally save visualization
    Returns per-video result dict.
    """
    print(f"\n  Processing: {video_name} ({stroke})")

    # frames_dir is already the full path to the video subfolder
    # (taken directly from folder_path in stroke_detection_results.json)
    # e.g. .../breaststroke_testing/breaststroke_front_S05
    video_folder = Path(frames_dir)
    if not video_folder.exists():
        print(f"    [WARN] Folder not found: {video_folder}")
        return {"error": f"folder not found: {video_folder}", "stroke": stroke}

    all_files = sorted(video_folder.glob("*.jpg"))
    if not all_files:
        all_files = sorted(video_folder.glob("*.png"))

    video_files = list(all_files)

    if not video_files:
        print(f"    [WARN] No frames found in {video_folder}")
        return {"error": "no frames found", "stroke": stroke}

    selected = select_frames(video_files, stroke)
    print(f"    Frames: {len(video_files)} total → {len(selected)} selected")

    model_id = BREASTSTROKE_MODEL if stroke == "breaststroke" else BUTTERFLY_MODEL
    frame_results = []
    api_errors = 0

    for frame_path in selected:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        b64 = encode_image_b64(str(frame_path))
        response = call_roboflow_keypoint(b64, model_id)

        if response is None:
            api_errors += 1
            continue

        keypoints    = parse_keypoints(response, w, h)
        measurements = compute_measurements(keypoints)

        frame_results.append({
            "frame": frame_path.name,
            "keypoints": {
                kp: {
                    "x": keypoints[kp]["x"],
                    "y": keypoints[kp]["y"],
                    "confidence": keypoints[kp]["confidence"],
                    "visible": keypoints[kp]["visible"]
                }
                for kp in KEYPOINT_NAMES
            },
            "measurements": measurements
        })

    if not frame_results:
        return {"error": "all API calls failed", "stroke": stroke,
                "api_errors": api_errors}

    # ── Aggregate per-video stats ─────────────────────────────────────────────
    valid_frames = [f for f in frame_results if f["measurements"]["valid"]]
    asym_vals = [f["measurements"]["shoulder_asymmetry"]
                 for f in valid_frames if f["measurements"]["shoulder_asymmetry"] is not None]
    width_vals = [f["measurements"]["shoulder_width"]
                  for f in valid_frames if f["measurements"]["shoulder_width"] is not None]
    offset_vals = [f["measurements"]["head_offset"]
                   for f in valid_frames if f["measurements"]["head_offset"] is not None]

    summary = {
        "mean_shoulder_asymmetry": float(np.mean(asym_vals)) if asym_vals else None,
        "mean_shoulder_width":     float(np.mean(width_vals)) if width_vals else None,
        "mean_head_offset":        float(np.mean(offset_vals)) if offset_vals else None,
        "frames_processed":        len(frame_results),
        "frames_valid":            len(valid_frames),
        "api_errors":              api_errors
    }

    print(f"    Valid frames: {len(valid_frames)}/{len(frame_results)}")
    if asym_vals:
        print(f"    Mean shoulder asymmetry: {summary['mean_shoulder_asymmetry']:.2f}px")
    if width_vals:
        print(f"    Mean shoulder width:     {summary['mean_shoulder_width']:.2f}px")

    return {
        "stroke": stroke,
        "frames": frame_results,
        "summary": summary
    }


def save_results(results: dict):
    """Save pose estimation results to Drive."""
    os.makedirs(os.path.dirname(POSE_RESULTS_PATH), exist_ok=True)
    with open(POSE_RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"\n✅ Pose results saved → {POSE_RESULTS_PATH}")


def run_pose_estimation():
    """Main entry point for File 2."""
    print("=" * 60)
    print("FILE 2: Pose Estimation")
    print("=" * 60)

    # Load File 1 results
    print("\nLoading stroke detection results...")
    stroke_results = load_stroke_results()
    print(f"  Found {len(stroke_results)} videos")

    video_dirs = get_video_dirs_by_stroke(stroke_results)

    all_pose_results = {}
    for video_name, (stroke, frames_dir) in video_dirs.items():
        result = process_video_frames(video_name, stroke, frames_dir)
        all_pose_results[video_name] = result

    save_results(all_pose_results)

    # Quick summary
    print("\n── Summary ──────────────────────────────────────────")
    for vname, res in all_pose_results.items():
        if "error" in res:
            print(f"  {vname}: ERROR — {res['error']}")
        else:
            s = res.get("summary", {})
            print(f"  {vname} ({res['stroke']}): "
                  f"{s.get('frames_valid', 0)}/{s.get('frames_processed', 0)} valid frames")
    print("=" * 60)

    return all_pose_results


if __name__ == "__main__":
    run_pose_estimation()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2: Pose Estimation

Loading stroke detection results...
  Found 7 videos

  Processing: breaststroke_front_S05 (breaststroke)
    Frames: 308 total → 247 selected


KeyboardInterrupt: 

In [7]:
"""
File 2: pose_estimation.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Extracts 7 upper-body keypoints per frame using fine-tuned Roboflow models.
Reads stroke detection results from Drive. Saves pose results to Drive.

CHANGE FROM PREVIOUS VERSION:
- Reads/writes from Google Drive paths, not /content/
- Runs Roboflow on PREPROCESSED frames from per-video subfolders:
    breaststroke_testing/{video_name}/frame_000000.jpg ...
    butterfly_testing (1)/{video_name}/frame_000000.jpg ...
"""

import os
import json
import base64
import requests
import numpy as np
import cv2
from pathlib import Path
from google.colab import drive

# ─── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ─── Paths (all Drive-based so they survive session restarts) ──────────────────
STROKE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/stroke_detection_results.json"
POSE_RESULTS_PATH   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results.json"
VIZ_DIR             = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations"

BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"
BUTTERFLY_TEST_DIR    = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"

# ─── Roboflow config ───────────────────────────────────────────────────────────
API_KEY        = "2gU1ewQS0rfbADpO4tCb"
WORKSPACE      = "habibas-workspace-jcdgt"
BREASTSTROKE_MODEL = "swimmer-breaststroke-front1/2"
BUTTERFLY_MODEL    = "swimmer-butterfly-front1/5"
ROBOFLOW_INFER_URL = "https://detect.roboflow.com/{model}"

# ─── Settings ─────────────────────────────────────────────────────────────────
SAMPLE_SIZE         = 300
KEYPOINT_THRESHOLD  = 0.70
KEYPOINT_NAMES      = ["head", "left_shoulder", "right_shoulder",
                       "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

# Visualization colors
COLOR_CONFIDENT  = (0, 255, 0)    # green
COLOR_OCCLUDED   = (128, 128, 128) # grey
COLOR_SKELETON   = (0, 0, 0)       # black
HALO_COLOR       = (0, 200, 0)

# Skeleton connections (index pairs into KEYPOINT_NAMES)
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]


def load_stroke_results():
    """Load File 1 output from Drive."""
    if not os.path.exists(STROKE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Stroke results not found at {STROKE_RESULTS_PATH}. "
            "Run File 1 (stroke_detection_cnn.py) first."
        )
    with open(STROKE_RESULTS_PATH, 'r') as f:
        return json.load(f)


def get_video_dirs_by_stroke(stroke_results: dict) -> dict:
    """
    Map each video name to its preprocessed frame subfolder and stroke type.
    Reads 'detected_stroke' from File 1 output.
    Builds the folder path using our own directory constants (not folder_path
    from File 1, which may have stale/renamed paths).
    Returns: {video_name: (stroke_type, video_subfolder_path)}
    """
    mapping = {}
    for video_name, result in stroke_results.items():
        stroke = result.get("detected_stroke", "unknown")
        if stroke == "breaststroke":
            folder_path = os.path.join(BREASTSTROKE_TEST_DIR, video_name)
            mapping[video_name] = (stroke, folder_path)
        elif stroke == "butterfly":
            folder_path = os.path.join(BUTTERFLY_TEST_DIR, video_name)
            mapping[video_name] = (stroke, folder_path)
        else:
            print(f"  [SKIP] {video_name} — unknown stroke '{stroke}'")
    return mapping


def select_frames(all_frames: list, stroke: str) -> list:
    """
    Select a representative subset of frames based on stroke type.
    Breaststroke: middle 80% (skip first/last 10%)
    Butterfly:    last 45% (skip first 55% — entry phase)
    Then sample up to SAMPLE_SIZE evenly.
    """
    n = len(all_frames)
    if stroke == "breaststroke":
        start = int(n * 0.10)
        end   = int(n * 0.90)
    else:  # butterfly
        start = int(n * 0.10)
        end   = int(n * 0.90)

    subset = all_frames[start:end]
    if len(subset) <= SAMPLE_SIZE:
        return subset

    # Evenly spaced sample
    indices = np.linspace(0, len(subset) - 1, SAMPLE_SIZE, dtype=int)
    return [subset[i] for i in indices]


def encode_image_b64(image_path: str) -> str:
    """Read image file and return base64 string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_roboflow_keypoint(image_b64: str, model_id: str) -> dict | None:
    """
    Call Roboflow REST API (NOT inference-sdk — broken in Colab).
    Returns raw response JSON or None on failure.
    """
    url = f"https://detect.roboflow.com/{model_id}?api_key={API_KEY}"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    try:
        resp = requests.post(url, data=image_b64, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"    [API ERROR] {e}")
        return None


def parse_keypoints(api_response: dict, img_w: int, img_h: int) -> dict:
    """
    Extract 7 keypoints from Roboflow response.
    Returns dict: {keypoint_name: {x, y, confidence, visible}}
    Falls back to estimated positions for missing keypoints.
    """
    keypoints = {}

    predictions = api_response.get("predictions", [])
    if not predictions:
        return {kp: {"x": None, "y": None, "confidence": 0.0, "visible": False}
                for kp in KEYPOINT_NAMES}

    # Take the highest-confidence prediction (the swimmer)
    best = max(predictions, key=lambda p: p.get("confidence", 0))
    raw_kps = {kp.get("class", "").lower(): kp
               for kp in best.get("keypoints", [])}

    for kp_name in KEYPOINT_NAMES:
        if kp_name in raw_kps:
            kp = raw_kps[kp_name]
            conf = float(kp.get("confidence", 0.0))
            x    = float(kp.get("x", 0))
            y    = float(kp.get("y", 0))
            keypoints[kp_name] = {
                "x": x, "y": y,
                "confidence": conf,
                "visible": conf >= KEYPOINT_THRESHOLD
            }
        else:
            keypoints[kp_name] = {
                "x": None, "y": None,
                "confidence": 0.0,
                "visible": False
            }

    return keypoints


def compute_measurements(keypoints: dict) -> dict:
    """
    Compute the 3 biomechanical measurements used for injury pattern detection.
    Returns dict with measurements and validity flags.
    """
    ls = keypoints.get("left_shoulder")
    rs = keypoints.get("right_shoulder")
    hd = keypoints.get("head")

    measurements = {
        "shoulder_asymmetry": None,     # abs(left_shoulder_y - right_shoulder_y)
        "shoulder_width": None,         # abs(left_shoulder_x - right_shoulder_x)
        "head_offset": None,            # abs(head_x - shoulder_midpoint_x)
        "valid": False
    }

    if ls["visible"] and rs["visible"] and ls["x"] is not None:
        measurements["shoulder_asymmetry"] = abs(ls["y"] - rs["y"])
        measurements["shoulder_width"]     = abs(ls["x"] - rs["x"])
        measurements["valid"] = True

        if hd["visible"] and hd["x"] is not None:
            mid_x = (ls["x"] + rs["x"]) / 2
            measurements["head_offset"] = abs(hd["x"] - mid_x)

    return measurements


def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """
    Draw glowing teal skeleton overlay matching the reference aesthetic:
    - Dark semi-transparent overlay on the frame
    - Multi-layer glowing cyan dots for confident keypoints
    - Grey dots for occluded keypoints
    - Bright glowing cyan lines for skeleton connections
    """
    # ── Dark overlay to deepen background like underwater feel ────────────────
    overlay = image.copy()
    cv2.rectangle(overlay, (0, 0), (image.shape[1], image.shape[0]),
                  (10, 15, 20), -1)
    img = cv2.addWeighted(image, 0.55, overlay, 0.45, 0)

    # Teal/cyan color palette (BGR)
    CYAN_BRIGHT  = (220, 240, 0)    # bright cyan-yellow core
    CYAN_MID     = (200, 220, 20)   # mid glow ring
    CYAN_OUTER   = (120, 180, 30)   # soft outer halo
    CYAN_GLOW    = (60,  140, 10)   # very soft bloom
    LINE_COLOR   = (180, 210, 15)   # glowing line color
    LINE_GLOW    = (60,  130, 5)    # line bloom
    OCCLUDED     = (80,  80,  80)   # grey for low-confidence

    kp_coords = {}

    # ── Collect visible keypoint positions ────────────────────────────────────
    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # ── Draw skeleton lines first (under dots) ────────────────────────────────
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            # Only draw if both endpoints are visible
            if kp_coords[n1][2] and kp_coords[n2][2]:
                # Outer bloom line
                cv2.line(img, p1, p2, LINE_GLOW,  5, cv2.LINE_AA)
                # Inner bright line
                cv2.line(img, p1, p2, LINE_COLOR, 2, cv2.LINE_AA)

    # ── Draw keypoint dots on top ─────────────────────────────────────────────
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            # 4-layer glow: bloom → outer → mid → bright core
            cv2.circle(img, (x, y), 16, CYAN_GLOW,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 11, CYAN_OUTER,  -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 7,  CYAN_MID,    -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 4,  CYAN_BRIGHT, -1, cv2.LINE_AA)
            # Bright white center pinpoint
            cv2.circle(img, (x, y), 2,  (255, 255, 255), -1, cv2.LINE_AA)
        else:
            # Occluded: simple grey dot
            cv2.circle(img, (x, y), 5, OCCLUDED, -1, cv2.LINE_AA)

    return img


def process_video_frames(video_name: str, stroke: str, frames_dir: str) -> dict:
    """
    Process a single video's frames:
    1. List and select frames
    2. Call Roboflow for each frame
    3. Parse keypoints + compute measurements
    4. Optionally save visualization
    Returns per-video result dict.
    """
    print(f"\n  Processing: {video_name} ({stroke})")

    # frames_dir is already the full path to the video subfolder
    # (taken directly from folder_path in stroke_detection_results.json)
    # e.g. .../breaststroke_testing/breaststroke_front_S05
    video_folder = Path(frames_dir)
    if not video_folder.exists():
        print(f"    [WARN] Folder not found: {video_folder}")
        return {"error": f"folder not found: {video_folder}", "stroke": stroke}

    all_files = sorted(video_folder.glob("*.jpg"))
    if not all_files:
        all_files = sorted(video_folder.glob("*.png"))

    video_files = list(all_files)

    if not video_files:
        print(f"    [WARN] No frames found in {video_folder}")
        return {"error": "no frames found", "stroke": stroke}

    selected = select_frames(video_files, stroke)
    print(f"    Frames: {len(video_files)} total → {len(selected)} selected")

    model_id = BREASTSTROKE_MODEL if stroke == "breaststroke" else BUTTERFLY_MODEL
    frame_results = []
    api_errors = 0

    for frame_path in selected:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        b64 = encode_image_b64(str(frame_path))
        response = call_roboflow_keypoint(b64, model_id)

        if response is None:
            api_errors += 1
            continue

        keypoints    = parse_keypoints(response, w, h)
        measurements = compute_measurements(keypoints)

        frame_results.append({
            "frame": frame_path.name,
            "keypoints": {
                kp: {
                    "x": keypoints[kp]["x"],
                    "y": keypoints[kp]["y"],
                    "confidence": keypoints[kp]["confidence"],
                    "visible": keypoints[kp]["visible"]
                }
                for kp in KEYPOINT_NAMES
            },
            "measurements": measurements
        })

    if not frame_results:
        return {"error": "all API calls failed", "stroke": stroke,
                "api_errors": api_errors}

    # ── Aggregate per-video stats ─────────────────────────────────────────────
    valid_frames = [f for f in frame_results if f["measurements"]["valid"]]
    asym_vals = [f["measurements"]["shoulder_asymmetry"]
                 for f in valid_frames if f["measurements"]["shoulder_asymmetry"] is not None]
    width_vals = [f["measurements"]["shoulder_width"]
                  for f in valid_frames if f["measurements"]["shoulder_width"] is not None]
    offset_vals = [f["measurements"]["head_offset"]
                   for f in valid_frames if f["measurements"]["head_offset"] is not None]

    summary = {
        "mean_shoulder_asymmetry": float(np.mean(asym_vals)) if asym_vals else None,
        "mean_shoulder_width":     float(np.mean(width_vals)) if width_vals else None,
        "mean_head_offset":        float(np.mean(offset_vals)) if offset_vals else None,
        "frames_processed":        len(frame_results),
        "frames_valid":            len(valid_frames),
        "api_errors":              api_errors
    }

    print(f"    Valid frames: {len(valid_frames)}/{len(frame_results)}")
    if asym_vals:
        print(f"    Mean shoulder asymmetry: {summary['mean_shoulder_asymmetry']:.2f}px")
    if width_vals:
        print(f"    Mean shoulder width:     {summary['mean_shoulder_width']:.2f}px")

    return {
        "stroke": stroke,
        "frames": frame_results,
        "summary": summary
    }


def save_results(results: dict):
    """Save pose estimation results to Drive."""
    os.makedirs(os.path.dirname(POSE_RESULTS_PATH), exist_ok=True)
    with open(POSE_RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"\n✅ Pose results saved → {POSE_RESULTS_PATH}")


def run_pose_estimation():
    """Main entry point for File 2."""
    print("=" * 60)
    print("FILE 2: Pose Estimation")
    print("=" * 60)

    # Load File 1 results
    print("\nLoading stroke detection results...")
    stroke_results = load_stroke_results()
    print(f"  Found {len(stroke_results)} videos")

    video_dirs = get_video_dirs_by_stroke(stroke_results)

    all_pose_results = {}
    for video_name, (stroke, frames_dir) in video_dirs.items():
        result = process_video_frames(video_name, stroke, frames_dir)
        all_pose_results[video_name] = result

    save_results(all_pose_results)

    # Quick summary
    print("\n── Summary ──────────────────────────────────────────")
    for vname, res in all_pose_results.items():
        if "error" in res:
            print(f"  {vname}: ERROR — {res['error']}")
        else:
            s = res.get("summary", {})
            print(f"  {vname} ({res['stroke']}): "
                  f"{s.get('frames_valid', 0)}/{s.get('frames_processed', 0)} valid frames")
    print("=" * 60)

    return all_pose_results


if __name__ == "__main__":
    run_pose_estimation()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2: Pose Estimation

Loading stroke detection results...
  Found 7 videos

  Processing: breaststroke_front_S05 (breaststroke)
    Frames: 308 total → 247 selected
    Valid frames: 13/247
    Mean shoulder asymmetry: 1.62px
    Mean shoulder width:     61.85px

  Processing: breaststroke_front_S06 (breaststroke)
    Frames: 234 total → 187 selected
    Valid frames: 34/187
    Mean shoulder asymmetry: 3.62px
    Mean shoulder width:     55.79px

  Processing: breaststroke_front_S07 (breaststroke)
    Frames: 190 total → 152 selected
    Valid frames: 26/152
    Mean shoulder asymmetry: 1.27px
    Mean shoulder width:     48.65px

  Processing: breaststroke_front_S08 (breaststroke)
    Frames: 153 total → 122 selected
    Valid frames: 7/122
    Mean shoulder asymmetry: 0.43px
    Mean shoulder width:     42.57px

  Processing: butterfly_front_S07 (butter

In [9]:
"""
File 2b: visualization.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Standalone visualization script.
Reads pose_estimation_results.json (File 2 output) and the preprocessed frames,
draws glowing teal skeleton overlays on every valid frame, and saves them to Drive.

Run this AFTER File 2 has completed successfully.

Output:
  Models/visualizations/{video_name}/{frame_name}.jpg
"""

import os
import json
import cv2
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
POSE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results.json"
VIZ_DIR           = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations"

BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"
BUTTERFLY_TEST_DIR    = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"

# ─── Keypoint config ───────────────────────────────────────────────────────────
KEYPOINT_NAMES = ["head", "left_shoulder", "right_shoulder",
                  "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """
    Draw glowing teal skeleton overlay matching the reference aesthetic:
    - Dark semi-transparent overlay for underwater feel
    - Multi-layer glowing cyan dots for confident keypoints
    - Grey dots for occluded keypoints
    - Bright glowing cyan lines for skeleton connections
    """
    # Dark overlay for underwater feel
    overlay = image.copy()
    cv2.rectangle(overlay, (0, 0), (image.shape[1], image.shape[0]),
                  (10, 15, 20), -1)
    img = cv2.addWeighted(image, 0.55, overlay, 0.45, 0)

    # Teal/cyan color palette (BGR)
    CYAN_BRIGHT = (220, 240, 0)
    CYAN_MID    = (200, 220, 20)
    CYAN_OUTER  = (120, 180, 30)
    CYAN_GLOW   = (60,  140, 10)
    LINE_COLOR  = (180, 210, 15)
    LINE_GLOW   = (60,  130, 5)
    OCCLUDED    = (80,  80,  80)

    kp_coords = {}

    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # Draw skeleton lines first (under dots)
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            if kp_coords[n1][2] and kp_coords[n2][2]:
                cv2.line(img, p1, p2, LINE_GLOW,  5, cv2.LINE_AA)
                cv2.line(img, p1, p2, LINE_COLOR, 2, cv2.LINE_AA)

    # Draw keypoint dots on top
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            cv2.circle(img, (x, y), 3, CYAN_GLOW,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 2, CYAN_OUTER,  -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 3, CYAN_MID,    -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 1, CYAN_BRIGHT, -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 1, (255, 255, 255), -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 2, OCCLUDED, -1, cv2.LINE_AA)

    return img


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def get_frame_path(video_name: str, stroke: str, frame_name: str) -> str:
    """Build full path to a preprocessed frame."""
    if stroke == "breaststroke":
        return os.path.join(BREASTSTROKE_TEST_DIR, video_name, frame_name)
    else:
        return os.path.join(BUTTERFLY_TEST_DIR, video_name, frame_name)


def run_visualization():
    print("=" * 60)
    print("FILE 2b: Skeleton Visualization")
    print("=" * 60)

    if not os.path.exists(POSE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Pose results not found at {POSE_RESULTS_PATH}. "
            "Run File 2 (pose_estimation.py) first."
        )

    with open(POSE_RESULTS_PATH, 'r') as f:
        pose_results = json.load(f)

    total_saved = 0

    for video_name, result in pose_results.items():
        if "error" in result:
            print(f"\n  [SKIP] {video_name} — {result['error']}")
            continue

        stroke = result.get("stroke", "unknown")
        frames = result.get("frames", [])

        # Sort frames by frame name to ensure correct temporal order
        frames_sorted = sorted(frames, key=lambda f: f.get("frame", ""))

        viz_video_dir = os.path.join(VIZ_DIR, video_name)
        os.makedirs(viz_video_dir, exist_ok=True)

        saved = 0
        print(f"\n  {video_name} ({stroke}) — {len(frames_sorted)} valid frames")

        for frame_data in frames_sorted:
            frame_name = frame_data.get("frame")
            keypoints  = frame_data.get("keypoints", {})

            # Load the original preprocessed frame
            frame_path = get_frame_path(video_name, stroke, frame_name)
            if not os.path.exists(frame_path):
                print(f"    [WARN] Frame not found: {frame_path}")
                continue

            img = cv2.imread(frame_path)
            if img is None:
                continue

            # Draw and save
            viz = draw_skeleton(img, keypoints)
            save_path = os.path.join(viz_video_dir, frame_name)
            cv2.imwrite(save_path, viz)
            saved += 1

        print(f"    Saved {saved} visualizations → {viz_video_dir}")
        total_saved += saved

    print(f"\n✅ Total visualizations saved: {total_saved}")
    print(f"   Location: {VIZ_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    run_visualization()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2b: Skeleton Visualization

  breaststroke_front_S05 (breaststroke) — 247 valid frames
    Saved 247 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke_front_S05

  breaststroke_front_S06 (breaststroke) — 187 valid frames
    Saved 187 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke_front_S06

  breaststroke_front_S07 (breaststroke) — 152 valid frames
    Saved 152 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke_front_S07

  breaststroke_front_S08 (breaststroke) — 122 valid frames
    Saved 122 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke_front_S08

  butterfly_front_S07 (butterfly) — 162 valid frames
    Saved 162 visuali

**Only butterfly**

In [2]:
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [1]:
"""
File 2 (Butterfly Only): pose_estimation_butterfly_only.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Same as pose_estimation.py but runs ONLY on butterfly videos
using the newly retrained model: swimmer-butterfly-front1/5
"""

import os
import json
import base64
import requests
import numpy as np
import cv2
from pathlib import Path
from google.colab import drive

# ─── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
STROKE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/stroke_detection_results.json"
POSE_RESULTS_PATH   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results.json"

BUTTERFLY_TEST_DIR  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"

# ─── Roboflow config ───────────────────────────────────────────────────────────
API_KEY          = "2gU1ewQS0rfbADpO4tCb"
BUTTERFLY_MODEL  = "swimmer-butterfly-front1/5"   # newly retrained model

# ─── Settings ─────────────────────────────────────────────────────────────────
SAMPLE_SIZE        = 300
KEYPOINT_THRESHOLD = 0.70
KEYPOINT_NAMES     = ["head", "left_shoulder", "right_shoulder",
                      "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),
    (1, 2),
    (1, 3), (3, 5),
    (2, 4), (4, 6),
]


def load_stroke_results():
    """Load File 1 output from Drive."""
    if not os.path.exists(STROKE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Stroke results not found at {STROKE_RESULTS_PATH}. "
            "Run File 1 first."
        )
    with open(STROKE_RESULTS_PATH, 'r') as f:
        return json.load(f)


def load_existing_pose_results():
    """Load existing pose results so we can merge butterfly results into them."""
    if os.path.exists(POSE_RESULTS_PATH):
        with open(POSE_RESULTS_PATH, 'r') as f:
            return json.load(f)
    return {}


def select_frames(all_frames: list) -> list:
    """Select middle 80% of frames, up to SAMPLE_SIZE evenly spaced."""
    n = len(all_frames)
    start = int(n * 0.10)
    end   = int(n * 0.90)
    subset = all_frames[start:end]
    if len(subset) <= SAMPLE_SIZE:
        return subset
    indices = np.linspace(0, len(subset) - 1, SAMPLE_SIZE, dtype=int)
    return [subset[i] for i in indices]


def encode_image_b64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_roboflow_keypoint(image_b64: str) -> dict | None:
    url = f"https://detect.roboflow.com/{BUTTERFLY_MODEL}?api_key={API_KEY}"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    try:
        resp = requests.post(url, data=image_b64, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"    [API ERROR] {e}")
        return None


def parse_keypoints(api_response: dict) -> dict:
    """Extract keypoints from Roboflow response. Trust confidence only — no geometric overrides."""
    keypoints = {}
    predictions = api_response.get("predictions", [])
    if not predictions:
        return {kp: {"x": None, "y": None, "confidence": 0.0, "visible": False}
                for kp in KEYPOINT_NAMES}

    best = max(predictions, key=lambda p: p.get("confidence", 0))
    raw_kps = {kp.get("class", "").lower(): kp
               for kp in best.get("keypoints", [])}

    for kp_name in KEYPOINT_NAMES:
        if kp_name in raw_kps:
            kp   = raw_kps[kp_name]
            conf = float(kp.get("confidence", 0.0))
            x    = float(kp.get("x", 0))
            y    = float(kp.get("y", 0))
            keypoints[kp_name] = {
                "x": x, "y": y,
                "confidence": conf,
                "visible": conf >= KEYPOINT_THRESHOLD
            }
        else:
            keypoints[kp_name] = {
                "x": None, "y": None,
                "confidence": 0.0,
                "visible": False
            }
    return keypoints


def compute_measurements(keypoints: dict) -> dict:
    ls = keypoints.get("left_shoulder")
    rs = keypoints.get("right_shoulder")
    hd = keypoints.get("head")

    measurements = {
        "shoulder_asymmetry": None,
        "shoulder_width": None,
        "head_offset": None,
        "valid": False
    }

    if ls["visible"] and rs["visible"] and ls["x"] is not None:
        measurements["shoulder_asymmetry"] = abs(ls["y"] - rs["y"])
        measurements["shoulder_width"]     = abs(ls["x"] - rs["x"])
        measurements["valid"] = True
        if hd["visible"] and hd["x"] is not None:
            mid_x = (ls["x"] + rs["x"]) / 2
            measurements["head_offset"] = abs(hd["x"] - mid_x)

    return measurements


def process_video(video_name: str) -> dict:
    print(f"\n  Processing: {video_name} (butterfly)")

    video_folder = Path(BUTTERFLY_TEST_DIR) / video_name
    if not video_folder.exists():
        print(f"    [WARN] Folder not found: {video_folder}")
        return {"error": f"folder not found: {video_folder}", "stroke": "butterfly"}

    # Sort frames by name to ensure correct temporal order (frame_000000 → end)
    all_files = sorted(video_folder.glob("*.jpg"))
    if not all_files:
        all_files = sorted(video_folder.glob("*.png"))

    if not all_files:
        print(f"    [WARN] No frames found in {video_folder}")
        return {"error": "no frames found", "stroke": "butterfly"}

    selected = select_frames(list(all_files))
    print(f"    Frames: {len(all_files)} total → {len(selected)} selected")

    frame_results = []
    api_errors    = 0

    for frame_path in selected:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue

        b64      = encode_image_b64(str(frame_path))
        response = call_roboflow_keypoint(b64)

        if response is None:
            api_errors += 1
            continue

        keypoints    = parse_keypoints(response)
        measurements = compute_measurements(keypoints)

        frame_results.append({
            "frame": frame_path.name,
            "keypoints": {
                kp: {
                    "x": keypoints[kp]["x"],
                    "y": keypoints[kp]["y"],
                    "confidence": keypoints[kp]["confidence"],
                    "visible": keypoints[kp]["visible"]
                }
                for kp in KEYPOINT_NAMES
            },
            "measurements": measurements
        })

    if not frame_results:
        return {"error": "all API calls failed", "stroke": "butterfly",
                "api_errors": api_errors}

    valid_frames = [f for f in frame_results if f["measurements"]["valid"]]
    asym_vals    = [f["measurements"]["shoulder_asymmetry"] for f in valid_frames
                    if f["measurements"]["shoulder_asymmetry"] is not None]
    width_vals   = [f["measurements"]["shoulder_width"] for f in valid_frames
                    if f["measurements"]["shoulder_width"] is not None]
    offset_vals  = [f["measurements"]["head_offset"] for f in valid_frames
                    if f["measurements"]["head_offset"] is not None]

    summary = {
        "mean_shoulder_asymmetry": float(np.mean(asym_vals)) if asym_vals else None,
        "mean_shoulder_width":     float(np.mean(width_vals)) if width_vals else None,
        "mean_head_offset":        float(np.mean(offset_vals)) if offset_vals else None,
        "frames_processed":        len(frame_results),
        "frames_valid":            len(valid_frames),
        "api_errors":              api_errors
    }

    print(f"    Valid frames: {len(valid_frames)}/{len(frame_results)}")
    if asym_vals:
        print(f"    Mean shoulder asymmetry: {summary['mean_shoulder_asymmetry']:.2f}px")
    if width_vals:
        print(f"    Mean shoulder width:     {summary['mean_shoulder_width']:.2f}px")

    return {"stroke": "butterfly", "frames": frame_results, "summary": summary}


def run_pose_estimation_butterfly():
    print("=" * 60)
    print("FILE 2 (Butterfly Only): Pose Estimation")
    print(f"Model: {BUTTERFLY_MODEL}")
    print("=" * 60)

    # Load File 1 results to find butterfly videos
    stroke_results = load_stroke_results()
    butterfly_videos = [
        name for name, res in stroke_results.items()
        if res.get("detected_stroke") == "butterfly"
    ]
    print(f"\n  Found {len(butterfly_videos)} butterfly videos: {butterfly_videos}")

    # Load existing pose results to preserve breaststroke results
    all_pose_results = load_existing_pose_results()
    print(f"  Loaded existing pose results ({len(all_pose_results)} videos) — breaststroke kept as-is")

    # Process only butterfly videos
    for video_name in butterfly_videos:
        result = process_video(video_name)
        all_pose_results[video_name] = result  # overwrite butterfly only

    # Save merged results
    os.makedirs(os.path.dirname(POSE_RESULTS_PATH), exist_ok=True)
    with open(POSE_RESULTS_PATH, 'w') as f:
        json.dump(all_pose_results, f, indent=2, default=str)
    print(f"\n✅ Results saved → {POSE_RESULTS_PATH}")

    print("\n── Summary ──────────────────────────────────────────")
    for vname, res in all_pose_results.items():
        if "error" in res:
            print(f"  {vname}: ERROR — {res['error']}")
        else:
            s = res.get("summary", {})
            print(f"  {vname} ({res['stroke']}): "
                  f"{s.get('frames_valid', 0)}/{s.get('frames_processed', 0)} valid frames")
    print("=" * 60)

    return all_pose_results


if __name__ == "__main__":
    run_pose_estimation_butterfly()

MessageError: Error: credential propagation was unsuccessful

**final pose**

In [11]:
"""
File 2c: visualization_butterfly.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Butterfly-only visualization script.
Reads pose_estimation_results.json, filters low confidence frames,
draws green skeleton overlay and saves to:
  Models/visualizations/butterfly/test/{video_name}/{frame_name}.jpg

Run AFTER pose_estimation_butterfly_only.py has completed.
"""

import os
import json
import cv2
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
POSE_RESULTS_PATH  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results.json"
BUTTERFLY_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"
VIZ_DIR            = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test"

# ─── Keypoint config ───────────────────────────────────────────────────────────
KEYPOINT_NAMES = ["head", "left_shoulder", "right_shoulder",
                  "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]

def compute_confidence_threshold(pose_results: dict, stroke: str = "butterfly") -> float:
    """
    Automatically compute confidence threshold from the data.
    Collects all non-zero average keypoint confidences, then sets the threshold
    at mean - 1 standard deviation to keep only clearly high-confidence frames.
    """
    all_confs = []
    for video_name, result in pose_results.items():
        if result.get("stroke") != stroke or "error" in result:
            continue
        for frame_data in result.get("frames", []):
            keypoints = frame_data.get("keypoints", {})
            kp_confs  = [
                kp["confidence"] for kp in keypoints.values()
                if kp.get("confidence") is not None
            ]
            if kp_confs:
                avg = sum(kp_confs) / len(kp_confs)
                if avg > 0.0:  # ignore completely empty frames
                    all_confs.append(avg)

    if not all_confs:
        return 0.75  # fallback

    arr    = np.array(all_confs)
    thresh = float(np.mean(arr) - np.std(arr))
    thresh = round(max(thresh, 0.50), 2)  # never go below 0.50

    print(f"  Non-zero frames: {len(arr)} | mean={np.mean(arr):.3f} | "
          f"std={np.std(arr):.3f} | threshold={thresh:.2f}")
    print(f"  Auto confidence threshold: {thresh:.2f} "
          f"(from {len(arr)} non-zero frames across all {stroke} videos)")
    return thresh


# ══════════════════════════════════════════════════════════════════════════════
# FILTER
# ══════════════════════════════════════════════════════════════════════════════

def filter_low_confidence_frames(frames: list, threshold: float) -> list:
    """
    Filter out frames where average keypoint confidence < threshold.
    Returns only frames with avg keypoint confidence >= threshold.
    """
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        kp_confs  = [
            kp["confidence"] for kp in keypoints.values()
            if kp.get("confidence") is not None
        ]
        if not kp_confs:
            continue
        avg_conf = sum(kp_confs) / len(kp_confs)
        if avg_conf >= threshold:
            filtered.append(frame_data)
    return filtered


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """Solid green dots scaled by shoulder width, with thick green skeleton lines."""
    img = image.copy()

    GREEN_BRIGHT = (0, 255, 0)
    GREEN_DARK   = (0, 180, 0)
    LINE_GREEN   = (0, 220, 0)
    OCCLUDED     = (80, 80, 80)

    kp_coords = {}
    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # Fixed small dot radius — no dynamic scaling
    dot_radius = 3

    # Lines first
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            if kp_coords[n1][2] and kp_coords[n2][2]:
                cv2.line(img, p1, p2, LINE_GREEN, 1, cv2.LINE_AA)

    # Dots on top
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            cv2.circle(img, (x, y), dot_radius, GREEN_DARK,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 2,          GREEN_BRIGHT, -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 2, OCCLUDED, -1, cv2.LINE_AA)

    return img


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run_visualization_butterfly():
    print("=" * 60)
    print("FILE 2c: Butterfly Visualization")
    print(f"Output: {VIZ_DIR}/{{video_name}}/{{frame_name}}.jpg")
    print("=" * 60)

    if not os.path.exists(POSE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Pose results not found at {POSE_RESULTS_PATH}. "
            "Run pose_estimation_butterfly_only.py first."
        )

    with open(POSE_RESULTS_PATH, 'r') as f:
        pose_results = json.load(f)

    # ── Step 1: Auto-compute confidence threshold from all butterfly frames ────
    print("\nComputing confidence threshold automatically...")
    threshold = compute_confidence_threshold(pose_results, stroke="butterfly")
    print(f"  Using threshold: {threshold:.2f}")

    total_saved = 0

    # ── Step 2: Filter and visualize ──────────────────────────────────────────
    for video_name, result in pose_results.items():
        if result.get("stroke") != "butterfly":
            continue
        if "error" in result:
            print(f"\n  [SKIP] {video_name} — {result['error']}")
            continue

        frames = result.get("frames", [])

        # Sort by frame name for correct temporal order
        frames_sorted = sorted(frames, key=lambda f: f.get("frame", ""))

        # Filter out low confidence frames using auto threshold
        frames_filtered = filter_low_confidence_frames(frames_sorted, threshold)

        print(f"\n  {video_name}: {len(frames_sorted)} total → "
              f"{len(frames_filtered)} after confidence filter (>= {threshold:.2f})")

        viz_video_dir = os.path.join(VIZ_DIR, video_name)
        os.makedirs(viz_video_dir, exist_ok=True)

        saved = 0
        for frame_data in frames_filtered:
            frame_name = frame_data.get("frame")
            keypoints  = frame_data.get("keypoints", {})

            # Skip frames where both shoulders not visible
            ls = keypoints.get("left_shoulder", {})
            rs = keypoints.get("right_shoulder", {})
            if not (ls.get("visible") and rs.get("visible")):
                continue

            frame_path = os.path.join(BUTTERFLY_TEST_DIR, video_name, frame_name)
            if not os.path.exists(frame_path):
                print(f"    [WARN] Frame not found: {frame_path}")
                continue

            img = cv2.imread(frame_path)
            if img is None:
                continue

            viz       = draw_skeleton(img, keypoints)
            save_path = os.path.join(viz_video_dir, frame_name)
            cv2.imwrite(save_path, viz)
            saved += 1

        print(f"    Saved {saved} visualizations → {viz_video_dir}")
        total_saved += saved

    print(f"\n✅ Total butterfly visualizations saved: {total_saved}")
    print(f"   Location: {VIZ_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    run_visualization_butterfly()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2c: Butterfly Visualization
Output: /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/{video_name}/{frame_name}.jpg

Computing confidence threshold automatically...
  Non-zero frames: 204 | mean=0.884 | std=0.168 | threshold=0.72
  Auto confidence threshold: 0.72 (from 204 non-zero frames across all butterfly videos)
  Using threshold: 0.72

  butterfly_front_S07: 162 total → 52 after confidence filter (>= 0.72)
    Saved 52 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/butterfly_front_S07

  butterfly_front_S08: 179 total → 76 after confidence filter (>= 0.72)
    Saved 76 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/butterfly_front_S08

  butterfly_front_S09: 136 total → 54 after confid